# Basic checks for LSTM daily features

This notebook reads the saved `daily_features_long` Delta output and runs the most important basic checks for the 21 model features:

1. Every sampled sequence has exactly 90 distinct days.
2. Features contain no null, NaN, or infinite values.
3. Zero rate, negative rate, approximate distinct count, mean, standard deviation, and key percentiles.
4. Binary, proportion, and non-negative features satisfy their expected ranges.

No charts are created and no data is written.

In [ ]:
from pyspark.sql import functions as F

DAILY_FEATURE_PATH = (
    'abfss://ml-artifact-insider-us@dsapdafazprdadls1.dfs.core.windows.net/'
    'mrm/ins_us_nms/output/insider_us_nms_lstm_daily_features_v1'
)

# Approximately 1/1000 of samples. All 90 days of a selected sample are retained.
# Use 500 for more data or 2000 for a faster, smaller check.
SAMPLE_DENOMINATOR = 1000
WINDOW_DAYS = 90
SAMPLE_KEY = ['login_id', 'acct_nbr', 'sample_date']

FEATURE_COLS = [
    'daily_inquiries',
    'daily_maintenances',
    'daily_prop_inquiries',
    'daily_prop_maintenances',
    'daily_after_hours_touches',
    'daily_emp_prop_acct_seqs',
    'daily_avg_gap_minutes',
    'daily_acct_balance',
    'acct_balance_missing',
    'daily_balance_change',
    'daily_status_change',
    'daily_sequence_duration_seconds',
    'balance_std_30d',
    'employee_daily_avg_touches_per_account',
    'daily_acct_balance_touch_deviation',
    'daily_emp_touch_seq_ratio',
    'daily_stddev_acct_balance_emp',
    'daily_acct_touch_seq_ratio',
    'daily_contact_info_changes',
    'daily_emp_behavior_deviation',
    'daily_address_changes',
]

BINARY_FEATURES = [
    'acct_balance_missing',
    'daily_status_change',
]

PROPORTION_FEATURES = [
    'daily_prop_inquiries',
    'daily_prop_maintenances',
    'daily_emp_prop_acct_seqs',
]

NON_NEGATIVE_FEATURES = [
    'daily_inquiries',
    'daily_maintenances',
    'daily_after_hours_touches',
    'daily_avg_gap_minutes',
    'daily_sequence_duration_seconds',
    'balance_std_30d',
    'employee_daily_avg_touches_per_account',
    'daily_emp_touch_seq_ratio',
    'daily_stddev_acct_balance_emp',
    'daily_acct_touch_seq_ratio',
    'daily_contact_info_changes',
    'daily_address_changes',
]

assert len(FEATURE_COLS) == 21

## 1. Load data and select complete sample sequences

Sampling by the three-column sample key ensures that a selected sample keeps all of its 90 daily rows. The hash is deterministic, so rerunning with the same denominator selects the same samples.

In [ ]:
daily_features_long = spark.read.format('delta').load(DAILY_FEATURE_PATH)

required_columns = SAMPLE_KEY + ['calendar_date', 'sequence_day'] + FEATURE_COLS
missing_columns = [c for c in required_columns if c not in daily_features_long.columns]
assert not missing_columns, f'Missing required columns: {missing_columns}'

sample_hash = F.pmod(
    F.xxhash64(*[F.coalesce(F.col(c).cast('string'), F.lit('NULL')) for c in SAMPLE_KEY]),
    F.lit(SAMPLE_DENOMINATOR),
)

feature_sample = (
    daily_features_long
    .filter(sample_hash == 0)
    .select(*required_columns)
    .cache()
)

## 2. Check the 90-day sequence structure

`bad_sequences` must be zero. This one check detects missing days, duplicate days, incorrect row counts, and sequence-day ranges other than 0 through 89.

In [ ]:
sequence_quality_by_sample = (
    feature_sample
    .groupBy(*SAMPLE_KEY)
    .agg(
        F.count('*').alias('rows'),
        F.countDistinct('calendar_date').alias('distinct_calendar_days'),
        F.countDistinct('sequence_day').alias('distinct_sequence_days'),
        F.min('sequence_day').alias('min_sequence_day'),
        F.max('sequence_day').alias('max_sequence_day'),
    )
    .withColumn(
        'is_bad',
        (F.col('rows') != WINDOW_DAYS)
        | (F.col('distinct_calendar_days') != WINDOW_DAYS)
        | (F.col('distinct_sequence_days') != WINDOW_DAYS)
        | (F.col('min_sequence_day') != 0)
        | (F.col('max_sequence_day') != WINDOW_DAYS - 1),
    )
)

sequence_quality_summary = sequence_quality_by_sample.agg(
    F.count('*').alias('sampled_sequences'),
    F.sum('rows').alias('sampled_daily_rows'),
    F.sum(F.col('is_bad').cast('long')).alias('bad_sequences'),
)

display(sequence_quality_summary)

## 3. Basic distribution checks for all 21 features

Important columns in the result:

- `null_rate`, `nan_rate`, `infinite_rate`: must normally be zero before LSTM training.
- `zero_rate`: identifies sparse daily features.
- `negative_rate`: helps verify which features contain negative values. Negative values are valid for balance changes and deviation features.
- `approx_distinct_values`: a value near one identifies a constant feature.
- `p01`, `p50`, `p95`, `p99`, and `max`: reveal skew and extreme outliers without requiring a chart.

In [ ]:
# Convert the 21 wide feature columns into a temporary feature/value structure.
feature_values = (
    feature_sample
    .select(
        F.explode(
            F.array(*[
                F.struct(
                    F.lit(c).alias('feature'),
                    F.col(c).cast('double').alias('value'),
                )
                for c in FEATURE_COLS
            ])
        ).alias('feature_value')
    )
    .select('feature_value.*')
    .withColumn('is_nan', F.isnan('value'))
    .withColumn('is_infinite', F.abs('value') == F.lit(float('inf')))
    .withColumn(
        'valid_value',
        F.when(
            F.col('value').isNotNull()
            & ~F.col('is_nan')
            & ~F.col('is_infinite'),
            F.col('value'),
        ),
    )
    .cache()
)

distribution_summary = (
    feature_values
    .groupBy('feature')
    .agg(
        F.count('*').alias('sampled_rows'),
        F.count('valid_value').alias('valid_rows'),
        F.sum(F.col('value').isNull().cast('long')).alias('null_rows'),
        F.sum(F.col('is_nan').cast('long')).alias('nan_rows'),
        F.sum(F.col('is_infinite').cast('long')).alias('infinite_rows'),
        F.sum((F.col('valid_value') == 0).cast('long')).alias('zero_rows'),
        F.sum((F.col('valid_value') < 0).cast('long')).alias('negative_rows'),
        F.approx_count_distinct('valid_value').alias('approx_distinct_values'),
        F.mean('valid_value').alias('mean'),
        F.stddev_pop('valid_value').alias('stddev'),
        F.min('valid_value').alias('min'),
        F.percentile_approx(
            'valid_value', [0.01, 0.50, 0.95, 0.99], 10000
        ).alias('quantiles'),
        F.max('valid_value').alias('max'),
    )
    .withColumn('null_rate', F.col('null_rows') / F.col('sampled_rows'))
    .withColumn('nan_rate', F.col('nan_rows') / F.col('sampled_rows'))
    .withColumn('infinite_rate', F.col('infinite_rows') / F.col('sampled_rows'))
    .withColumn('zero_rate', F.col('zero_rows') / F.col('valid_rows'))
    .withColumn('negative_rate', F.col('negative_rows') / F.col('valid_rows'))
    .select(
        'feature', 'sampled_rows', 'valid_rows',
        'null_rate', 'nan_rate', 'infinite_rate', 'zero_rate', 'negative_rate',
        'approx_distinct_values', 'mean', 'stddev', 'min',
        F.col('quantiles')[0].alias('p01'),
        F.col('quantiles')[1].alias('p50'),
        F.col('quantiles')[2].alias('p95'),
        F.col('quantiles')[3].alias('p99'),
        'max',
    )
    .orderBy('feature')
)

display(distribution_summary)

## 4. Basic business-range checks

Expected rules:

- Missing/status flags must be either 0 or 1.
- Proportion features must be between 0 and 1.
- Counts, durations, standard deviations, and touch/sequence ratios must not be negative.
- Inquiry proportion plus maintenance proportion must not exceed 1.

`invalid_rows` should be zero for every rule. Balance, balance change, and deviation features are not forced to be non-negative because legitimate negative values are possible.

In [ ]:
checked_features = BINARY_FEATURES + PROPORTION_FEATURES + NON_NEGATIVE_FEATURES

range_check_rows = (
    feature_values
    .filter(F.col('feature').isin(checked_features))
    .withColumn(
        'rule',
        F.when(F.col('feature').isin(BINARY_FEATURES), F.lit('value must be 0 or 1'))
        .when(F.col('feature').isin(PROPORTION_FEATURES), F.lit('value must be between 0 and 1'))
        .otherwise(F.lit('value must be non-negative')),
    )
    .withColumn(
        'is_invalid',
        F.when(
            F.col('valid_value').isNull(),
            F.lit(False),
        )
        .when(
            F.col('feature').isin(BINARY_FEATURES),
            ~F.col('valid_value').isin(0.0, 1.0),
        )
        .when(
            F.col('feature').isin(PROPORTION_FEATURES),
            (F.col('valid_value') < 0) | (F.col('valid_value') > 1),
        )
        .otherwise(F.col('valid_value') < 0),
    )
    .groupBy('feature', 'rule')
    .agg(
        F.count('valid_value').alias('checked_rows'),
        F.sum(F.col('is_invalid').cast('long')).alias('invalid_rows'),
    )
)

proportion_sum_check = (
    feature_sample
    .agg(
        F.count('*').alias('checked_rows'),
        F.sum(
            (
                F.col('daily_prop_inquiries')
                + F.col('daily_prop_maintenances')
                > F.lit(1.0 + 1e-12)
            ).cast('long')
        ).alias('invalid_rows'),
    )
    .withColumn('feature', F.lit('daily_prop_inquiries + daily_prop_maintenances'))
    .withColumn('rule', F.lit('sum must not exceed 1'))
    .select('feature', 'rule', 'checked_rows', 'invalid_rows')
)

range_check_summary = (
    range_check_rows
    .unionByName(proportion_sum_check)
    .withColumn('invalid_rate', F.col('invalid_rows') / F.col('checked_rows'))
    .orderBy(F.desc('invalid_rows'), 'feature')
)

display(range_check_summary)

## Minimum acceptance criteria

Before using these features in the LSTM, check that:

- `bad_sequences = 0`.
- Every feature has `null_rate = nan_rate = infinite_rate = 0`.
- Every range rule has `invalid_rows = 0`.
- Features with `approx_distinct_values = 1` are reviewed because they provide no variation in this dataset.
- Very high `zero_rate` is expected for sparse event features, but it should be reviewed.
- A very large gap between `p99` and `max` indicates extreme outliers that may need investigation or scaling.

These checks use a deterministic sample for speed. The original feature-generation notebook already performs full-data null and 90-day-length assertions before saving.